# M5: дневной прогноз и WRMSSE

Цель третьего ноутбука — получить сопоставимый с M5 Forecasting — Accuracy результат, а затем улучшать его. Прогнозируем продажи по дням сразу на 28 дней из одной даты. Основная метрика — WRMSSE: меньшее значение лучше.

[Первый ноутбук](01_exploration.ipynb) помог проверить данные и недельную задачу на 100 товарах. Во [втором](02_scaling.ipynb) обучили модель для всего CA_1, проверили признаки и временные окна. Используем эти наработки при подготовке дневных признаков. Недельные MAE и WAPE сохраняются как результат отдельной версии проекта.

**Первый расчёт здесь — WRMSSE простого дневного прогноза.** После этого появится точка отсчёта для CatBoost, LightGBM и новых признаков.

Начальные ячейки подготовлены, но пока не выполнены. Конкурсный балл ещё не рассчитан.


## 0. Окружение и пути

Используем `.venv` проекта и уже загруженные исходные файлы. Эта ячейка проверяет интерпретатор и наличие данных. Повторно скачивать архив или запускать недельные обучения не нужно.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [current_dir, *current_dir.parents]
     if (path / "requirements.txt").is_file() and (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Открой ноутбук из demand-planner или notebooks.")

if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Выбери Python из .venv проекта.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / "data" / "raw"
RESULTS = PROJECT_ROOT / "results" / "m5_wrmsse"
HORIZON = 28
RANDOM_STATE = 42

required_files = [
    "calendar.csv",
    "sales_train_validation.csv",
    "sell_prices.csv",
]
missing_files = [name for name in required_files if not (DATA_RAW / name).is_file()]

if missing_files:
    raise FileNotFoundError(f"Не найдены исходные файлы: {missing_files}")

pd.set_option("display.max_columns", 30)

print(f"Python: {sys.version.split()[0]}")
print(f"Проект: {PROJECT_ROOT}")
print(f"Горизонт: {HORIZON} дней из одной даты")


## 1. Даты и правила проверки

| Период | Дни M5 | Для чего используем |
|---|---|---|
| До 24 апреля 2016 | `d_1` — `d_1913` | История для воспроизведения public-постановки |
| 25 апреля — 22 мая 2016 | `d_1914` — `d_1941` | Период public leaderboard, теперь используем для разработки |
| 23 мая — 19 июня 2016 | `d_1942` — `d_1969` | Период private leaderboard, резервируем для итоговой проверки |

Апрель–май уже смотрели во втором ноутбуке для CA_1. Независимым финальным тестом следующей версии он больше не является. До итоговой проверки выбираем варианты на нескольких исторических 28-дневных окнах. Ответы private-периода понадобятся из опубликованных организаторами данных; в исходных CSV архива истории до `d_1941` их нет.

Для каждого окна фиксируем одну дату прогноза. На следующие 28 дней используем только доступную к ней историю и заранее известные внешние признаки. Реальные продажи внутри прогнозируемых 28 дней в лаги не подставляем.


In [ ]:
calendar = pd.read_csv(DATA_RAW / "calendar.csv", parse_dates=["date"])
calendar_by_day = calendar.set_index("d")

periods = pd.DataFrame([
    {"period": "История", "first_day": "d_1", "last_day": "d_1913"},
    {"period": "Public / разработка", "first_day": "d_1914", "last_day": "d_1941"},
    {"period": "Private / финальная проверка", "first_day": "d_1942", "last_day": "d_1969"},
])

periods["start"] = periods["first_day"].map(calendar_by_day["date"])
periods["end"] = periods["last_day"].map(calendar_by_day["date"])
periods["days"] = (periods["end"] - periods["start"]).dt.days + 1

assert periods.loc[periods["period"].ne("История"), "days"].eq(HORIZON).all()
display(periods.set_index("period"))


## 2. Что входит в полный скор

В M5 есть 30 490 рядов «товар — магазин». Прогнозы этих рядов суммируются до магазина, категории, штата и других групп. Всего оцениваются 42 840 рядов на 12 уровнях.

Ниже оставляем в памяти только идентификаторы, чтобы проверить состав иерархии. Длинную таблицу всех дневных продаж пока не создаём. Для быстрых модельных экспериментов можно начинать с CA_1, но полный конкурсный WRMSSE относится ко всем десяти магазинам.


In [ ]:
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

series_metadata = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=identifier_columns,
    dtype="string",
)

assert not series_metadata.duplicated(["item_id", "store_id"]).any()
assert len(series_metadata) == 30_490
assert series_metadata["store_id"].nunique() == 10

M5_LEVELS = {
    "Total": [],
    "State": ["state_id"],
    "Store": ["store_id"],
    "Category": ["cat_id"],
    "Department": ["dept_id"],
    "State / Category": ["state_id", "cat_id"],
    "State / Department": ["state_id", "dept_id"],
    "Store / Category": ["store_id", "cat_id"],
    "Store / Department": ["store_id", "dept_id"],
    "Item": ["item_id"],
    "Item / State": ["item_id", "state_id"],
    "Item / Store": ["item_id", "store_id"],
}

hierarchy = pd.DataFrame([
    {
        "level": level,
        "series": (
            series_metadata.groupby(columns, observed=True).ngroups
            if columns else 1
        ),
    }
    for level, columns in M5_LEVELS.items()
]).set_index("level")

assert int(hierarchy["series"].sum()) == 42_840
display(hierarchy)
print(f"Всего оцениваемых рядов: {hierarchy['series'].sum():,}")


## 3. WRMSSE

Сначала для каждого ряда считаем RMSSE:

$$
\mathrm{RMSSE}_i =
\sqrt{\frac{\frac{1}{28}\sum_{h=1}^{28}(y_{i,T+h}-\hat y_{i,T+h})^2}
{\frac{1}{n_i-1}\sum_{t=2}^{n_i}(y_{i,t}-y_{i,t-1})^2}}.
$$

Знаменатель считаем по истории после удаления ведущих нулей до первой ненулевой продажи. Это масштаб обычных дневных изменений ряда. Числитель — средняя квадратичная ошибка прогноза на 28 дней.

Затем суммируем RMSSE с весами по денежным продажам последних 28 дней обучения. Веса внутри каждого уровня дают единицу; вклад каждого из 12 уровней равен одной двенадцатой. Цены используются для весов, целевая переменная остаётся количеством проданных единиц.

Следующий рабочий блок — реализация этого расчёта и сверка со справочным расчётом организаторов. Нужно отдельно проверить порядок рядов, агрегацию, масштабы и веса. Код метрики добавим следующим шагом.


## 4. Первый дневной бейзлайн

После реализации метрики сравним три простых прогноза, доступных из одной даты:

- повтор последней известной недели четыре раза;
- средние продажи соответствующего дня недели за последние четыре недели;
- повтор последнего известного блока из 28 дней.

Сохраняем общий WRMSSE и вклад каждого уровня. До первого запуска модели нужно знать, какой простой метод сильнее в этой постановке.


## 5. Улучшение прогноза

Порядок экспериментов:

1. Дневной CatBoost и проверка MAE, RMSE, Poisson и Tweedie как функций обучения.
2. Дневная сезонность, длинная история, прошлые продажи групп товаров, цена и календарь.
3. LightGBM на тех же строках и с теми же правилами доступности признаков.
4. Разные способы объединения магазинов и категорий при обучении.
5. Ансамбль и коррекция смещения по историческим проверочным прогнозам.
6. Полная оценка по всем магазинам и однократная проверка выбранного варианта на зарезервированном периоде.

У каждого эксперимента фиксируем признаки, параметры, дату прогноза, WRMSSE, время и изменение относительно бейзлайна. Число деревьев и параметры подбираем на исторических окнах. Результат на одном CA_1 отмечаем как результат подвыборки.

В резюме можно будет указать воспроизведённый WRMSSE и улучшение относительно бейзлайна с названием периода. Место на Kaggle указывается только при наличии подтверждённого результата на платформе.


## Источники

- [Материалы организаторов M5: данные, расчёт метрик и решения](https://github.com/Mcompetitions/M5-methods).
- [M5 Accuracy competition: результаты и описание оценки](https://statmodeling.stat.columbia.edu/wp-content/uploads/2021/10/M5_accuracy_competition.pdf).
- [Страница M5 Forecasting — Accuracy](https://www.kaggle.com/competitions/m5-forecasting-accuracy).
